# 🎬 Roblox Story Kit - Google Colab GPU Renderer

Offload the heavy 3D rendering to free Google Colab GPUs (T4 / L4).

> **Tip:** Before running, ensure you have enabled GPU acceleration:
> Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
# @title 1. Environment & GPU Check
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation
!pip install -q numpy pillow faster-whisper yt-dlp

In [ ]:
# @title 2. Download & Install Blender 5.2 (Linux x64)
import os, subprocess

BLENDER_TAR = "blender-5.2.4-linux-x64.tar.xz"
BLENDER_DIR = "/content/blender-5.2.4-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

if not os.path.exists(BLENDER_BIN):
    print("Downloading Blender 5.2.4 for Linux...")
    !wget -q -c https://download.blender.org/release/Blender5.2/blender-5.2.4-linux-x64.tar.xz
    print("Extracting Blender...")
    !tar -xf blender-5.2.4-linux-x64.tar.xz
    print("Blender ready!")
else:
    print("Blender already installed at:", BLENDER_BIN)

os.environ["BLENDER"] = BLENDER_BIN
!$BLENDER --version

In [ ]:
# @title 3. Clone / Update Repository
import os

# Set your repository URL here (or default to current project)
REPO_URL = "https://github.com/arjunmalpani/roblox-stories.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

if not os.path.exists("/content/roblox-stories"):
    !git clone -b $BRANCH $REPO_URL /content/roblox-stories
else:
    %cd /content/roblox-stories
    !git pull origin $BRANCH

%cd /content/roblox-stories
!ls -la

In [ ]:
# @title 4. Render 3D Video on GPU
import os
%cd /content/roblox-stories

VIDEO_SLUG = "hiccup_shrink" #@param ["hiccup_shrink", "sneeze_teleport", "laugh_float"] {allow-input: true}

print(f"--- Rendering 3D frames for '{VIDEO_SLUG}' in Blender ---")
!python3 tools/blend.py $VIDEO_SLUG final all

print(f"--- Finishing video (audio mix, sound effects, subtitles, muxing) ---")
!python3 tools/finish.py $VIDEO_SLUG

output_file = f"videos/{VIDEO_SLUG}/out/final.mp4"
if os.path.exists(output_file):
    print(f"SUCCESS! Rendered video at: {output_file} ({os.path.getsize(output_file)/(1024*1024):.1f} MB)")
else:
    print("Render finished. Check logs above if output was not generated.")

In [ ]:
# @title 5. Preview & Download Finished Video
from IPython.display import HTML
from base64 import b64encode
from google.colab import files
import os

mp4_path = f"/content/roblox-stories/videos/{VIDEO_SLUG}/out/final_phone.mp4"
if not os.path.exists(mp4_path):
    mp4_path = f"/content/roblox-stories/videos/{VIDEO_SLUG}/out/final.mp4"

if os.path.exists(mp4_path):
    # Preview in notebook
    mp4 = open(mp4_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="320" height="568" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
    
    # Download button
    print("Triggering file download to your browser...")
    files.download(f"/content/roblox-stories/videos/{VIDEO_SLUG}/out/final.mp4")
else:
    print("Video file not found. Please run step 4 first.")